# 01: Graphs on Shapes - Dual Representation

**Goal**: Implement the `GraphShape` class that maintains synchronized graph topology and geometric shapes

**Learning Objectives**:
- Understand the dual graph-shape representation
- Create graphs that "live on" geometric shapes
- Visualize topology and geometry simultaneously
- Validate graph-shape consistency

**Key Concept**: Every node in the graph has a corresponding Rectangle, and every edge represents spatial adjacency

## Setup

In [ ]:
# Add src to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Import shape primitives
from grammar.shapes import Point, Rectangle, Square

# Import graph-shape integration
from grammar.rules import GraphShape

# For graph analysis
import networkx as nx

# For visualization
try:
    from topologicpy.Plotly import Plotly
    from topologicpy.Cluster import Cluster
    TOPOLOGIC_AVAILABLE = True
except ImportError:
    print("⚠️  TopologicPy not installed")
    TOPOLOGIC_AVAILABLE = False

try:
    from pyvis.network import Network
    PYVIS_AVAILABLE = True
except ImportError:
    print("⚠️  pyvis not installed. Install with: pip install pyvis")
    PYVIS_AVAILABLE = False

print("✅ Imports successful")
print(f"   TopologicPy: {'✓' if TOPOLOGIC_AVAILABLE else '✗'}")
print(f"   pyvis: {'✓' if PYVIS_AVAILABLE else '✗'}")

---

## Part 1: Understanding GraphShape

The `GraphShape` class is the core innovation - it maintains **dual representation**:
- **Topology**: Graph with nodes and edges (abstract connectivity)
- **Geometry**: Rectangles with positions and dimensions (concrete layout)

These two representations stay **synchronized** - every graph operation updates geometry, and vice versa.

### 1.1: Creating a Simple GraphShape

In [ ]:
# Create a simple 3-node graph
# Layout: [A][B]
#         [C]   

shapes = {
    'A': Rectangle(5, 4, Point(0, 4)),
    'B': Rectangle(5, 4, Point(5, 4)),
    'C': Rectangle(5, 4, Point(0, 0))
}

edges = [
    ('A', 'B'),  # Horizontal adjacency
    ('A', 'C')   # Vertical adjacency
]

gs = GraphShape(shapes=shapes, edges=edges)

print("GraphShape created:")
print(f"  Nodes: {len(gs.shapes)}")
print(f"  Edges: {len(gs.edges)}")
print(f"  Total area: {gs.total_area():.1f}")
print()

# Inspect individual shapes
for node_id, rect in gs.shapes.items():
    print(f"  {node_id}: {rect.width}×{rect.height} at {rect.origin}")

### 1.2: Graph Properties

In [ ]:
# NetworkX integration - get graph structure
G = gs.to_networkx()

print("Graph analysis:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.2f}")
print(f"  Connected: {nx.is_connected(G)}")
print()

# Degree of each node
print("Node degrees:")
for node, degree in G.degree():
    print(f"  {node}: {degree} connection(s)")

---

## Part 2: Validation - Ensuring Consistency

The graph and shapes must be consistent:
1. **No overlaps**: Shapes shouldn't overlap (unless intentional)
2. **Edges match adjacencies**: Graph edges should correspond to geometric adjacencies
3. **Connectivity**: Graph should be connected (one component)

### 2.1: Overlap Detection

In [ ]:
# Check for overlapping shapes
overlaps = gs.find_overlaps()

if overlaps:
    print(f"⚠️  Found {len(overlaps)} overlaps:")
    for n1, n2 in overlaps:
        print(f"  {n1} ↔ {n2}")
else:
    print("✅ No overlapping shapes detected")

### 2.2: Adjacency Validation

In [ ]:
# Verify that edges correspond to actual adjacencies
print("Edge validation:")
for n1, n2 in gs.edges:
    rect1 = gs.shapes[n1]
    rect2 = gs.shapes[n2]
    adjacent = rect1.is_adjacent_to(rect2)
    
    status = "✅" if adjacent else "❌"
    print(f"  {status} ({n1}, {n2}): geometrically adjacent = {adjacent}")

# Find geometric adjacencies NOT in graph
missing_edges = gs.find_missing_adjacencies()
if missing_edges:
    print(f"\n⚠️  Found {len(missing_edges)} geometric adjacencies not in graph:")
    for n1, n2 in missing_edges:
        print(f"  {n1} ↔ {n2}")
else:
    print("\n✅ All geometric adjacencies represented in graph")

### 2.3: Comprehensive Validation

In [ ]:
# Run all validation checks
is_valid, issues = gs.validate()

if is_valid:
    print("✅ GraphShape is valid")
else:
    print(f"❌ GraphShape has {len(issues)} issue(s):")
    for issue in issues:
        print(f"  - {issue}")

---

## Part 3: Building Complex Layouts

Let's create more realistic floor plan layouts with proper room adjacencies.

### 3.1: Linear Layout (Hallway Apartment)

In [ ]:
# Create a linear apartment: Entrance -> Kitchen -> Living -> Bedroom
linear_shapes = {
    'Entrance': Rectangle(3, 4, Point(0, 0)),
    'Kitchen': Rectangle(5, 4, Point(3, 0)),
    'Living': Rectangle(6, 4, Point(8, 0)),
    'Bedroom': Rectangle(5, 4, Point(14, 0))
}

linear_edges = [
    ('Entrance', 'Kitchen'),
    ('Kitchen', 'Living'),
    ('Living', 'Bedroom')
]

linear_gs = GraphShape(shapes=linear_shapes, edges=linear_edges)

print("Linear apartment:")
print(f"  Total area: {linear_gs.total_area():.1f} m²")
print(f"  Dimensions: {linear_gs.bounding_box()}")
print(f"  Connected: {nx.is_connected(linear_gs.to_networkx())}")

# Validate
is_valid, issues = linear_gs.validate()
print(f"  Valid: {'✅' if is_valid else '❌'}")

### 3.2: Grid Layout (Open Plan Apartment)

In [ ]:
# Create a 2x2 grid layout
# [Living    ][Kitchen]
# [Bedroom1  ][Bedroom2]

grid_shapes = {
    'Living': Rectangle(6, 5, Point(0, 5)),
    'Kitchen': Rectangle(4, 5, Point(6, 5)),
    'Bedroom1': Rectangle(6, 5, Point(0, 0)),
    'Bedroom2': Rectangle(4, 5, Point(6, 0))
}

grid_edges = [
    ('Living', 'Kitchen'),   # Top row
    ('Bedroom1', 'Bedroom2'), # Bottom row
    ('Living', 'Bedroom1'),  # Left column
    ('Kitchen', 'Bedroom2')  # Right column
]

grid_gs = GraphShape(shapes=grid_shapes, edges=grid_edges)

print("Grid apartment:")
print(f"  Total area: {grid_gs.total_area():.1f} m²")
print(f"  Average degree: {sum(dict(grid_gs.to_networkx().degree()).values()) / len(grid_shapes):.1f}")

# Check connectivity
G = grid_gs.to_networkx()
print("\nConnectivity analysis:")
for node in G.nodes():
    neighbors = list(G.neighbors(node))
    print(f"  {node}: connected to {neighbors}")

### 3.3: Complex Layout (Realistic Apartment)

In [ ]:
# Create a realistic apartment layout
# Central corridor connecting multiple rooms

complex_shapes = {
    'Entrance': Rectangle(3, 3, Point(0, 4)),
    'Corridor': Rectangle(10, 2, Point(3, 5)),
    'Kitchen': Rectangle(4, 4, Point(3, 8)),
    'Living': Rectangle(6, 5, Point(7, 8)),
    'Bathroom': Rectangle(3, 3, Point(3, 1)),
    'Bedroom1': Rectangle(5, 4, Point(6, 0)),
    'Bedroom2': Rectangle(4, 3, Point(11, 0))
}

complex_edges = [
    ('Entrance', 'Corridor'),
    ('Corridor', 'Kitchen'),
    ('Corridor', 'Living'),
    ('Corridor', 'Bathroom'),
    ('Corridor', 'Bedroom1'),
    ('Kitchen', 'Living'),
    ('Bedroom1', 'Bedroom2')
]

complex_gs = GraphShape(shapes=complex_shapes, edges=complex_edges)

print("Complex apartment:")
print(f"  Rooms: {len(complex_shapes)}")
print(f"  Total area: {complex_gs.total_area():.1f} m²")
print(f"  Graph density: {nx.density(complex_gs.to_networkx()):.2f}")

# Find central nodes (highest degree)
G = complex_gs.to_networkx()
degrees = dict(G.degree())
central_node = max(degrees, key=degrees.get)
print(f"\nCentral room: {central_node} (degree {degrees[central_node]})")

---

## Part 4: Geometric Visualization

Visualize just the shapes using TopologicPy + Plotly

In [ ]:
if TOPOLOGIC_AVAILABLE:
    from grammar.shapes import rectangle_to_topologic_face
    
    # Convert complex apartment to TopologicPy faces
    faces = []
    for room_id, rect in complex_gs.shapes.items():
        # Add room label to metadata
        rect_labeled = rect.with_metadata(label=room_id)
        face = rectangle_to_topologic_face(rect_labeled)
        faces.append(face)
    
    cluster = Cluster.ByTopologies(faces)
    
    print("🎨 Generating geometric visualization...")
    
    Plotly.Show(
        cluster,
        renderer="notebook",
        width=900,
        height=700,
        camera=[0, 0, 40],
        target=[7, 5, 0],
        backgroundColor='white',
        faceColor='lightblue',
        faceOpacity=0.7,
        edgeColor='darkblue',
        edgeWidth=2,
        showFaces=True,
        showEdges=True,
        showVertices=False
    )
    
    # Export to HTML
    Path("viz_outputs").mkdir(exist_ok=True)
    Plotly.ExportToHTML(
        cluster,
        path="viz_outputs/01_complex_apartment_geometry.html",
        title="Complex Apartment - Geometric Layout"
    )
    
    print("✅ Exported to: viz_outputs/01_complex_apartment_geometry.html")
else:
    print("⚠️  Skipping visualization (TopologicPy not installed)")

---

## Part 5: Graph Visualization

Visualize the topology using pyvis network graph

In [ ]:
if PYVIS_AVAILABLE:
    # Create interactive network graph
    net = Network(
        height='600px',
        width='100%',
        bgcolor='#ffffff',
        font_color='black'
    )
    
    # Add nodes with room names
    for room_id, rect in complex_gs.shapes.items():
        # Color based on room type
        colors = {
            'Entrance': '#ff9999',
            'Corridor': '#cccccc',
            'Kitchen': '#ffcc99',
            'Living': '#99ccff',
            'Bedroom1': '#99ff99',
            'Bedroom2': '#99ff99',
            'Bathroom': '#cc99ff'
        }
        color = colors.get(room_id, '#dddddd')
        
        # Size based on area
        size = rect.area() * 2
        
        net.add_node(
            room_id,
            label=room_id,
            color=color,
            size=size,
            title=f"{room_id}<br>Area: {rect.area():.1f} m²<br>Dimensions: {rect.width}×{rect.height}"
        )
    
    # Add edges
    for n1, n2 in complex_gs.edges:
        net.add_edge(n1, n2, color='#666666')
    
    # Enable physics for interactive layout
    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -8000,
          "springLength": 150,
          "springConstant": 0.04
        }
      }
    }
    """)
    
    # Save
    net.save_graph("viz_outputs/01_complex_apartment_graph.html")
    print("✅ Graph visualization exported to: viz_outputs/01_complex_apartment_graph.html")
    print("   Open in browser to interact with the network")
else:
    print("⚠️  Skipping graph visualization (pyvis not installed)")

---

## Part 6: Combined Visualization (Overlay)

The ultimate goal - show both topology AND geometry together

### 6.1: Side-by-Side Comparison

In [ ]:
# For now, we have separate visualizations
# In Phase 2.1, we'll overlay graph edges on the Plotly geometry

print("📊 Dual Visualization Summary:")
print("="*60)
print("Geometric view (Plotly):")
print("  - Shows actual room positions and dimensions")
print("  - Interactive: pan, zoom, rotate")
print("  - File: viz_outputs/01_complex_apartment_geometry.html")
print()
print("Topological view (pyvis):")
print("  - Shows room connectivity (abstract graph)")
print("  - Interactive: drag nodes, physics simulation")
print("  - Node size ∝ room area")
print("  - File: viz_outputs/01_complex_apartment_graph.html")
print("="*60)

---

## Part 7: Factory Methods

Convenient ways to create GraphShapes

### 7.1: From Grid Subdivision

In [ ]:
# Create GraphShape from subdividing a base rectangle
base = Rectangle(20, 15, Point(0, 0))
gs_grid = GraphShape.from_grid(base, rows=3, cols=3)

print("Grid-based GraphShape:")
print(f"  Nodes: {len(gs_grid.shapes)}")
print(f"  Edges: {len(gs_grid.edges)}")
print(f"  Total area: {gs_grid.total_area():.1f}")
print(f"  Matches original: {abs(gs_grid.total_area() - base.area()) < 0.01}")

# Grid creates a mesh topology
G = gs_grid.to_networkx()
print(f"\n  Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.1f}")

### 7.2: From Horizontal Subdivision

In [ ]:
# Create GraphShape from horizontal split
base = Rectangle(30, 10, Point(0, 0))
gs_split = GraphShape.from_horizontal_split(
    base,
    ratios=[0.3, 0.4, 0.3],
    labels=['Entrance', 'Living', 'Bedroom']
)

print("Horizontal split GraphShape:")
print(f"  Nodes: {len(gs_split.shapes)}")
print(f"  Edges: {len(gs_split.edges)}")
print()

# Linear topology (chain graph)
for node_id, rect in gs_split.shapes.items():
    print(f"  {node_id}: {rect.width}×{rect.height} = {rect.area():.1f} m²")

---

## Part 8: Tests - GraphShape Validation

In [ ]:
print("="*70)
print("GRAPHSHAPE TEST SUITE")
print("="*70)
print()

tests_passed = 0
tests_total = 0

# Test 1: Area conservation
tests_total += 1
base = Rectangle(100, 100)
gs_test = GraphShape.from_grid(base, 4, 4)
if abs(gs_test.total_area() - base.area()) < 0.01:
    print("✅ Test 1: Area conservation in grid construction")
    tests_passed += 1
else:
    print("❌ Test 1: Area mismatch")

# Test 2: No overlaps in grid
tests_total += 1
overlaps = gs_test.find_overlaps()
if len(overlaps) == 0:
    print("✅ Test 2: Grid has no overlapping shapes")
    tests_passed += 1
else:
    print(f"❌ Test 2: Found {len(overlaps)} overlaps")

# Test 3: Graph connectivity
tests_total += 1
if nx.is_connected(gs_test.to_networkx()):
    print("✅ Test 3: Grid graph is connected")
    tests_passed += 1
else:
    print("❌ Test 3: Graph is disconnected")

# Test 4: Adjacency validation
tests_total += 1
valid_adjacencies = True
for n1, n2 in gs_test.edges:
    if not gs_test.shapes[n1].is_adjacent_to(gs_test.shapes[n2]):
        valid_adjacencies = False
        break

if valid_adjacencies:
    print("✅ Test 4: All edges correspond to geometric adjacencies")
    tests_passed += 1
else:
    print("❌ Test 4: Some edges don't match adjacencies")

# Test 5: Linear split topology
tests_total += 1
base = Rectangle(30, 10)
gs_linear = GraphShape.from_horizontal_split(base, [0.33, 0.34, 0.33])
G_linear = gs_linear.to_networkx()
# Linear graph should have exactly 2 nodes with degree 1 (endpoints)
endpoints = [n for n, d in G_linear.degree() if d == 1]
if len(endpoints) == 2:
    print("✅ Test 5: Linear split creates chain topology (2 endpoints)")
    tests_passed += 1
else:
    print(f"❌ Test 5: Expected 2 endpoints, got {len(endpoints)}")

# Test 6: Bounding box calculation
tests_total += 1
bbox = gs_test.bounding_box()
if bbox == (0, 0, 100, 100):
    print(f"✅ Test 6: Bounding box correct: {bbox}")
    tests_passed += 1
else:
    print(f"❌ Test 6: Bounding box mismatch: {bbox}")

print()
print("="*70)
print(f"RESULTS: {tests_passed}/{tests_total} tests passed")
print("="*70)

if tests_passed == tests_total:
    print("🎉 ALL TESTS PASSED!")
else:
    print(f"⚠️  {tests_total - tests_passed} test(s) failed")

---

## Summary

**What we learned**:
1. ✅ `GraphShape` maintains dual topology-geometry representation
2. ✅ Validation ensures consistency (no overlaps, edges match adjacencies)
3. ✅ Factory methods make creating common patterns easy
4. ✅ NetworkX integration enables graph analysis
5. ✅ Dual visualization (Plotly for geometry, pyvis for topology)

**Next steps**:
- `02_Transformation_Rules.ipynb` - Implement graph grammar rules
- Add graph overlays to geometric visualizations
- Implement rule-based transformations (split, merge)

**Architecture insight**: The dual representation is the foundation for graph grammars. When we transform the graph (split a node), the geometry automatically updates (subdivide the rectangle). This keeps topology and geometry synchronized.